### Import Libraries and Load Data

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

FILES_2021 = [
    r"BR_REPORTING_2021\BR_REPORTING_2021_0.csv",
    r"BR_REPORTING_2021\BR_REPORTING_2021_1.csv",
]
FILES_2023 = [
    r"BR_REPORTING_2023\BR_REPORTING_2023_0.csv",
    r"BR_REPORTING_2023\BR_REPORTING_2023_1.csv",
]

# Output directory for results
out_dir = Path("CSVs for Visualization")
out_dir.mkdir(parents=True, exist_ok=True)

# Configure pandas to display floats with 2 decimal places and commas as thousands separators
pd.options.display.float_format = "{:,.0f}".format

def read_br_csv(path: str, year: int) -> pd.DataFrame:
    df = pd.read_csv(
        path,
        sep=",",
        engine="python",
    )
    return df

# Load & append 2021
df_2021 = pd.concat([read_br_csv(p, 2021) for p in FILES_2021], ignore_index=True)
print("2021 rows:", f"{len(df_2021):,}")

# Load & append 2023
df_2023 = pd.concat([read_br_csv(p, 2023) for p in FILES_2023], ignore_index=True)
print("2023 rows:", f"{len(df_2023):,}")

# Combine years
df_all = pd.concat([df_2021, df_2023], ignore_index=True)
del df_2021, df_2023
print("-" * 20)
print("ALL rows:", f"{len(df_all):,}")

2021 rows: 1,829,496
2023 rows: 1,769,356
--------------------
ALL rows: 3,598,852


### National Totals

In [2]:
# Aggregation by year (REPORT CYCLE), national totals. 
national = (
    df_all.groupby("REPORT CYCLE", as_index=False)
          .agg(total_tons=("GENERATION TONS", "sum"), records=("GENERATION TONS", "size"))
          .sort_values("REPORT CYCLE")
)

# Format records with commas
national_display = national.copy()
national_display["records"] = national_display["records"].map("{:,}".format)

# Display the national totals by year
print("National totals by year:")
print("-" * 35)
print(national_display.to_string(index=False))

# Display total difference and percentage change from 2021 to 2023
tons_2021 = national.loc[national["REPORT CYCLE"] == 2021, "total_tons"].values[0]
tons_2023 = national.loc[national["REPORT CYCLE"] == 2023, "total_tons"].values[0]
difference = tons_2023 - tons_2021
percent_change = (difference / tons_2021) * 100
print("-" * 35)
print(f"Difference (tons): {difference:,.0f}")
print(f"Percentage change: {percent_change:.2f}%")  

National totals by year:
-----------------------------------
 REPORT CYCLE  total_tons   records
         2021  82,368,244 1,829,496
         2023  69,684,726 1,769,356
-----------------------------------
Difference (tons): -12,683,518
Percentage change: -15.40%


### State Totals

In [ ]:
# Aggregation by year and state
state_totals = (
    df_all.groupby(["REPORT CYCLE", "STATE NAME"], as_index=False)
          .agg(tons=("GENERATION TONS", "sum"))
)

# Pivot to have states as rows and years as columns, filling missing values with 0.0
state_pivot = (
    state_totals.pivot(index="STATE NAME", columns="REPORT CYCLE", values="tons")
                .fillna(0.0)
)

# Calculate absolute change (delta) and percentage change from 2021 to 2023
state_pivot["delta"] = state_pivot.get(2023, 0.0) - state_pivot.get(2021, 0.0)

base = state_pivot.get(2021, 0.0)

INCLUDE_ZERO_BASE = False  # Set to True to include states with 2021 = 0 in percentage change calculation

if INCLUDE_ZERO_BASE:
    # Allow divide-by-zero (will produce inf where 2021 = 0)
    state_pivot["pct_change"] = state_pivot["delta"] / base * 100
else:
    # Safe version (exclude 2021 = 0 from pct calc)
    state_pivot["pct_change"] = np.where(base > 0, state_pivot["delta"] / base * 100, np.nan)

# Sort by 2023 total, descending
state_pivot = state_pivot.sort_values(2023, ascending=False)

# Display copy formatting percent change
state_pivot_display = state_pivot.copy()
state_pivot_display["pct_change"] = state_pivot_display["pct_change"].map(
    lambda x: f"{x:.1f}%" if pd.notnull(x) else ""
)

# Sort by delta, descending (uncomment if you want to sort by absolute change instead)
# state_pivot_display = state_pivot_display.sort_values("delta", ascending=False)

# Sort by percentage change, descending (uncomment if you want to sort by percentage instead)
# state_pivot_display = state_pivot_display.sort_values("pct_change", ascending=False)

# Display the top 20 states
print("Top 20 states by total tons in 2023:")
print("-" * 60)
print(state_pivot_display.head(20))
print("-" * 60)

Top 20 states by total tons in 2023:
------------------------------------------------------------
REPORT CYCLE       2021       2023      delta pct_change
STATE NAME                                              
TEXAS        30,988,747 27,894,005 -3,094,742     -10.0%
NEW YORK     27,118,682 19,881,003 -7,237,679     -26.7%
LOUISIANA     5,498,513  4,732,693   -765,821     -13.9%
WYOMING       3,022,055  3,303,815    281,760       9.3%
OHIO          1,479,511  1,712,332    232,822      15.7%
MISSISSIPPI   2,145,316  1,264,540   -880,775     -41.1%
ALABAMA       1,208,047  1,200,919     -7,129      -0.6%
KANSAS        1,031,809  1,053,014     21,205       2.1%
TENNESSEE       935,014    897,966    -37,048      -4.0%
INDIANA         945,284    864,979    -80,305      -8.5%
CALIFORNIA      425,720    778,752    353,032      82.9%
NEW MEXICO        3,820    771,741    767,921   20102.7%
MISSOURI      1,110,783    741,744   -369,039     -33.2%
HAWAII          542,014    544,647      2,633  

: 

### Facility Concentration (Top Generators)

In [4]:
TOP_N = 20
YEARS = [2021, 2023]
DISPLAY_YEAR = 2023   # Change to 2021 when presenting

# Full facility table (ALL rows) for Tableau
facility_all = (
    df_all.groupby(
        ["REPORT CYCLE", "STATE NAME", "HANDLER ID", "HANDLER NAME"],
        as_index=False
    )
    .agg(tons=("GENERATION TONS", "sum"))
)

# Add national totals per year (for share calcs)
year_totals = (
    facility_all.groupby("REPORT CYCLE", as_index=False)
                .agg(year_total_tons=("tons", "sum"))
)

facility_all = facility_all.merge(year_totals, on="REPORT CYCLE", how="left")

# Shares for every row (numeric proportions; format as % in Tableau)
facility_all["national_share"] = np.where(
    facility_all["year_total_tons"] > 0,
    facility_all["tons"] / facility_all["year_total_tons"],
    np.nan
)

# Rank within each year
facility_all["Rank"] = (
    facility_all.groupby("REPORT CYCLE")["tons"]
                .rank(method="first", ascending=False)
                .astype(int)
)

# Cumulative share within year (sorted by tons)
facility_all = facility_all.sort_values(["REPORT CYCLE", "tons"], ascending=[True, False])
facility_all["cumulative_share"] = facility_all.groupby("REPORT CYCLE")["national_share"].cumsum()

# Storytelling view (Top N only) — for notebook printing
def show_top_facilities(year, df, top_n=20):
    view = (
        df[df["REPORT CYCLE"] == year]
        .sort_values("Rank")
        .head(top_n)
        .copy()
    )

    view["HANDLER NAME"] = (
        view["HANDLER NAME"]
        .astype(str)
        .str.slice(0, 40)
        .str.rstrip() + "…"
    )

    # Format shares for printing
    view["national_share"] = (view["national_share"] * 100).map(
        lambda x: f"{x:,.1f}%" if pd.notna(x) else ""
    )
    view["cumulative_share"] = (view["cumulative_share"] * 100).map(
        lambda x: f"{x:,.1f}%" if pd.notna(x) else ""
    )

    print(f"\nTop {top_n} Facilities in {year}")
    print("-" * 125)
    print(view[["REPORT CYCLE","STATE NAME","HANDLER ID","HANDLER NAME","tons","national_share","cumulative_share"]]
          .to_string(index=False))
    print("-" * 125)

    top_share = (
        df[df["REPORT CYCLE"] == year]
        .sort_values("Rank")["cumulative_share"]
        .iloc[top_n - 1] * 100
    )
    print(f"Top {top_n} facilities account for {top_share:,.1f}% of {year} generated tons.")
    return top_share

# Display selected year
show_top_facilities(DISPLAY_YEAR, facility_all, TOP_N)

# Print dual year summary
for y in YEARS:
    share_y = (
        facility_all[facility_all["REPORT CYCLE"] == y]
        .sort_values("Rank")["cumulative_share"]
        .iloc[TOP_N - 1] * 100
    )
    print(f"Top {TOP_N} share in {y}: {share_y:,.1f}%")
print("-" * 125)


Top 20 Facilities in 2023
-----------------------------------------------------------------------------------------------------------------------------
 REPORT CYCLE  STATE NAME   HANDLER ID                              HANDLER NAME      tons national_share cumulative_share
         2023    NEW YORK NYD002080034                        MPM SILICONES LLC… 6,583,145           9.4%             9.4%
         2023       TEXAS TXD001700806 ASCEND PERFORMANCE MATERIALS CHOCOLATE B… 6,341,330           9.1%            18.5%
         2023       TEXAS TXD007330202                         EASTMAN CHEMICAL… 4,024,747           5.8%            24.3%
         2023    NEW YORK NYR000179887                          GLOBALFOUNDRIES… 3,814,396           5.5%            29.8%
         2023    NEW YORK NYD980592497 EASTMAN KODAK CO & RED - ROCHESTER LLC A… 3,152,524           4.5%            34.3%
         2023     WYOMING WYD079959185           HF SINCLAIR PARCO REFINING LLC… 2,800,374           4.0%    

#### Facilities Appearing in Top N for Each Year

In [5]:
# Function to get top N facilities for a given year
def get_top_n(year):
    df = (
        facility_all[facility_all["REPORT CYCLE"] == year]
        .sort_values("tons", ascending=False)
        .head(TOP_N)
        .copy()
    )
    df = df[["STATE NAME", "HANDLER ID", "HANDLER NAME", "tons"]]
    df = df.rename(columns={"tons": f"tons_{year}"})
    return df


# Get top facilities for each year
top_2021 = get_top_n(2021)
top_2023 = get_top_n(2023)

# Find facilities that appear in both years
common = pd.merge(
    top_2021,
    top_2023,
    on=["STATE NAME", "HANDLER ID", "HANDLER NAME"],
    how="inner"
)

# Calculate change
common["delta"] = common["tons_2023"] - common["tons_2021"]
common["pct_change"] = np.where(
    common["tons_2021"] > 0,
    common["delta"] / common["tons_2021"] * 100,
    np.nan
)

# Sort by 2023 volume (largest first)
common = common.sort_values("tons_2023", ascending=False)

# Display copy (truncate names for readability)
common_display = common.copy()

common_display["HANDLER NAME"] = (
    common_display["HANDLER NAME"]
    .astype(str)
    .str.slice(0, 40)
    .str.rstrip() + "…"
)

# Format percentages
common_display["pct_change"] = common_display["pct_change"].map(
    lambda x: f"{x:.1f}%" if pd.notna(x) else ""
)

# Add 1-based rank column for display
common_display.insert(0, "Rank", range(1, len(common_display) + 1))

print("\nFacilities Appearing in Top 20 in Both 2021 and 2023")
print("-" * 120)

print(common_display.to_string(index=False))
print("-" * 120)


Facilities Appearing in Top 20 in Both 2021 and 2023
------------------------------------------------------------------------------------------------------------------------
 Rank  STATE NAME   HANDLER ID                              HANDLER NAME  tons_2021  tons_2023      delta pct_change
    1    NEW YORK NYD002080034                        MPM SILICONES LLC… 14,329,700  6,583,145 -7,746,555     -54.1%
    2       TEXAS TXD001700806 ASCEND PERFORMANCE MATERIALS CHOCOLATE B…  6,884,396  6,341,330   -543,066      -7.9%
    3       TEXAS TXD007330202                         EASTMAN CHEMICAL…  3,937,865  4,024,747     86,882       2.2%
    4    NEW YORK NYR000179887                          GLOBALFOUNDRIES…  4,491,820  3,814,396   -677,424     -15.1%
    5       TEXAS TXD059685339                       VALERO MCKEE PLANT…  2,272,205  2,191,075    -81,130      -3.6%
    6       TEXAS TXD008091290                 PASADENA REFINING SYSTEM…  3,277,994  2,091,455 -1,186,539     -36.2%
    7 

### Industry (NAICS) Totals

In [6]:
# Clean NAICS as strings (no commas, no decimals)
df_all["NAICS"] = (
    df_all["PRIMARY NAICS"]
    .astype(str)
    .str.replace(r"\.0$", "", regex=True)
)

# Aggregate generated tons by year and NAICS
naics_totals = (
    df_all.groupby(["REPORT CYCLE", "NAICS"], as_index=False)
          .agg(tons=("GENERATION TONS", "sum"))
)

# Pivot to compare 2021 vs 2023 (NUMERIC core table)
naics_pivot = (
    naics_totals.pivot(index="NAICS", columns="REPORT CYCLE", values="tons")
                .fillna(0.0)
)

# Change metrics (numeric)
naics_pivot["delta"] = naics_pivot.get(2023, 0.0) - naics_pivot.get(2021, 0.0)
base = naics_pivot.get(2021, 0.0)
naics_pivot["pct_change"] = np.where(base > 0, naics_pivot["delta"] / base * 100, np.nan)

# Composition share (numeric)
total_2021 = naics_pivot.get(2021, 0.0).sum()
total_2023 = naics_pivot.get(2023, 0.0).sum()
naics_pivot["share_2021"] = np.where(
    total_2021 > 0,
    naics_pivot.get(2021, 0.0) / total_2021 * 100,
    np.nan,
)
naics_pivot["share_2023"] = np.where(
    total_2023 > 0,
    naics_pivot.get(2023, 0.0) / total_2023 * 100,
    np.nan,
)

# Sort by 2023 total (largest industries first)
naics_pivot = naics_pivot.sort_values(2023, ascending=False)

# Display copy (formatted strings for percents)
naics_display = naics_pivot.head(TOP_N).reset_index().copy()
naics_display["pct_change"] = naics_display["pct_change"].map(
    lambda x: f"{x:.1f}%" if pd.notna(x) else ""
)
naics_display["share_2021"] = naics_display["share_2021"].map(
    lambda x: f"{x:.1f}%" if pd.notna(x) else ""
)
naics_display["share_2023"] = naics_display["share_2023"].map(
    lambda x: f"{x:.1f}%" if pd.notna(x) else ""
)

print(f"\nTop {TOP_N} NAICS Industries by Total Tons in 2023")
print("-" * 75)
print(naics_display.to_string(index=False))
print("-" * 75)


Top 20 NAICS Industries by Total Tons in 2023
---------------------------------------------------------------------------
 NAICS       2021       2023      delta pct_change share_2021 share_2023
325199 12,634,800 12,613,526    -21,274      -0.2%      15.3%      18.1%
324110  9,229,664  7,559,725 -1,669,938     -18.1%      11.2%      10.8%
325180  8,131,755  6,978,810 -1,152,944     -14.2%       9.9%      10.0%
325211 16,207,749  6,791,782 -9,415,967     -58.1%      19.7%       9.7%
334413  5,097,024  4,342,139   -754,885     -14.8%       6.2%       6.2%
 32411  3,985,909  4,240,351    254,442       6.4%       4.8%       6.1%
325992  3,652,975  3,237,091   -415,884     -11.4%       4.4%       4.6%
325194  3,338,485  2,957,123   -381,362     -11.4%       4.1%       4.2%
562211  2,741,225  2,340,940   -400,285     -14.6%       3.3%       3.4%
562998  1,579,852  1,956,184    376,332      23.8%       1.9%       2.8%
331492  2,062,388  1,902,219   -160,169      -7.8%       2.5%       2.7%
2

### Waste Code Group Totals

In [7]:
# Aggregate generated tons by year and waste code group
waste_totals = (
    df_all.groupby(["REPORT CYCLE", "WASTE CODE GROUP"], as_index=False)
          .agg(tons=("GENERATION TONS", "sum"))
)

# Pivot to compare 2021 vs 2023 (NUMERIC core table)
waste_pivot = (
    waste_totals.pivot(index="WASTE CODE GROUP", columns="REPORT CYCLE", values="tons")
                .fillna(0.0)
)

# Change metrics (numeric)
waste_pivot["delta"] = waste_pivot.get(2023, 0.0) - waste_pivot.get(2021, 0.0)
base = waste_pivot.get(2021, 0.0)
waste_pivot["pct_change"] = np.where(base > 0, waste_pivot["delta"] / base * 100, np.nan)

# Composition share (numeric)
total_2021 = waste_pivot.get(2021, 0.0).sum()
total_2023 = waste_pivot.get(2023, 0.0).sum()
waste_pivot["share_2021"] = np.where(
    total_2021 > 0,
    waste_pivot.get(2021, 0.0) / total_2021 * 100,
    np.nan,
)
waste_pivot["share_2023"] = np.where(
    total_2023 > 0,
    waste_pivot.get(2023, 0.0) / total_2023 * 100,
    np.nan,
)

# Sort by 2023 total (largest waste groups first)
waste_pivot = waste_pivot.sort_values(2023, ascending=False)

# Display copy (formatted strings for percents)
waste_display = waste_pivot.head(TOP_N).reset_index().copy()
waste_display["pct_change"] = waste_display["pct_change"].map(
    lambda x: f"{x:.1f}%" if pd.notna(x) else ""
)
waste_display["share_2021"] = waste_display["share_2021"].map(
    lambda x: f"{x:.1f}%" if pd.notna(x) else ""
)
waste_display["share_2023"] = waste_display["share_2023"].map(
    lambda x: f"{x:.1f}%" if pd.notna(x) else ""
)

print(f"\nTop {TOP_N} Waste Code Groups by Total Tons in 2023")
print("-" * 85)
print(waste_display.to_string(index=False))
print("-" * 85)


Top 20 Waste Code Groups by Total Tons in 2023
-------------------------------------------------------------------------------------
WASTE CODE GROUP       2021       2023      delta pct_change share_2021 share_2023
            F039 24,203,067 24,171,792    -31,275      -0.1%      29.5%      35.0%
         TCORICR 11,598,116  9,655,300 -1,942,816     -16.8%      14.1%      14.0%
            D018  9,710,109  8,491,697 -1,218,413     -12.5%      11.8%      12.3%
            D002 12,525,349  5,229,602 -7,295,747     -58.2%      15.2%       7.6%
            KXXX  4,357,174  4,508,685    151,511       3.5%       5.3%       6.5%
            TCMT  3,613,597  2,690,876   -922,722     -25.5%       4.4%       3.9%
            F1_5  1,476,921  1,942,294    465,373      31.5%       1.8%       2.8%
            K013  1,895,215  1,589,250   -305,965     -16.1%       2.3%       2.3%
            K061  1,447,114  1,435,140    -11,974      -0.8%       1.8%       2.1%
            DMIX  1,834,270  1,406,2

### Drivers of Change (State × Waste Code Group)

In [8]:
# Aggregate generated tons by year, state, and waste code group
drivers = (
    df_all.groupby(["REPORT CYCLE", "STATE NAME", "WASTE CODE GROUP"], as_index=False)
          .agg(tons=("GENERATION TONS", "sum"))
)

# Pivot to put 2021 and 2023 side-by-side (NUMERIC core table)
drivers_pivot = (
    drivers.pivot_table(
        index=["STATE NAME", "WASTE CODE GROUP"],
        columns="REPORT CYCLE",
        values="tons",
        aggfunc="sum",
        fill_value=0.0
    )
)

# Change metrics (numeric)
drivers_pivot["delta"] = drivers_pivot.get(2023, 0.0) - drivers_pivot.get(2021, 0.0)
base = drivers_pivot.get(2021, 0.0)
drivers_pivot["pct_change"] = np.where(base > 0, drivers_pivot["delta"] / base * 100, np.nan)

# Sort once (largest increases)
drivers_sorted_inc = drivers_pivot.sort_values("delta", ascending=False)

# Display increases (formatted)
drivers_inc_display = drivers_sorted_inc.head(TOP_N).reset_index().copy()
drivers_inc_display.insert(0, "Rank", range(1, len(drivers_inc_display) + 1))
drivers_inc_display["pct_change"] = drivers_inc_display["pct_change"].map(lambda x: f"{x:.1f}%" if pd.notna(x) else "")

print(f"\nTop {TOP_N} Drivers of Increase (State × Waste Group), 2021 → 2023")
print("-" * 75)
print(drivers_inc_display.to_string(index=False))
print("-" * 75)




Top 20 Drivers of Increase (State × Waste Group), 2021 → 2023
---------------------------------------------------------------------------
 Rank STATE NAME WASTE CODE GROUP      2021      2023     delta  pct_change
    1      TEXAS             F039 7,727,192 9,169,537 1,442,345       18.7%
    2       OHIO             KXXX    16,336   788,743   772,407     4728.2%
    3 NEW MEXICO             F1_5       140   767,005   766,866   549124.2%
    4       OHIO             U043         0   349,629   349,629            
    5    WYOMING             D018 3,013,123 3,301,282   288,159        9.6%
    6  TENNESSEE             D002   127,428   328,771   201,343      158.0%
    7 WASHINGTON              ICR        99   162,411   162,313   164653.8%
    8   ARKANSAS             LABP         1   136,276   136,276 25543699.0%
    9   NEW YORK             D040   279,790   357,538    77,748       27.8%
   10   NEW YORK             U_PS     2,051    68,324    66,274     3231.8%
   11    ALABAMA         

In [9]:
# Display decreases (formatted)
drivers_sorted_dec = drivers_pivot.sort_values("delta", ascending=True)

drivers_dec_display = drivers_sorted_dec.head(TOP_N).reset_index().copy()
drivers_dec_display.insert(0, "Rank", range(1, len(drivers_dec_display) + 1))
drivers_dec_display["pct_change"] = drivers_dec_display["pct_change"].map(lambda x: f"{x:.1f}%" if pd.notna(x) else "")

print(f"\nTop {TOP_N} Drivers of Decrease (State × Waste Group), 2021 → 2023")
print("-" * 80)
print(drivers_dec_display.to_string(index=False))
print("-" * 80)


Top 20 Drivers of Decrease (State × Waste Group), 2021 → 2023
--------------------------------------------------------------------------------
 Rank   STATE NAME WASTE CODE GROUP       2021       2023      delta pct_change
    1     NEW YORK             D002 11,158,861  4,383,195 -6,775,666     -60.7%
    2        TEXAS          TCORICR 10,857,832  9,025,450 -1,832,382     -16.9%
    3        TEXAS             D018  5,786,034  4,248,827 -1,537,207     -26.6%
    4  MISSISSIPPI             TCMT  2,054,552  1,173,221   -881,332     -42.9%
    5     ARKANSAS             F039    728,295     85,736   -642,558     -88.2%
    6     KENTUCKY             D002    618,248      5,707   -612,541     -99.1%
    7     NEW YORK             F039 13,252,706 12,745,816   -506,890      -3.8%
    8    LOUISIANA             KXXX  3,464,999  2,961,850   -503,149     -14.5%
    9        TEXAS             DMIX  1,301,016    909,478   -391,538     -30.1%
   10         OHIO             K011    353,347          

### CSV Exports for Visualization

In [46]:
# Write the dataframes to .csv files within the "CSVs for Visualization" folder for use in Tableau or other tools
out_dir = Path("CSVs for Visualization")
out_dir.mkdir(parents=True, exist_ok=True)

national.to_csv(out_dir / "national_totals.csv", index=False)
state_totals.to_csv(out_dir / "state_totals.csv", index=False)
state_pivot.reset_index().to_csv(out_dir / "state_totals_pivot.csv", index=False)
facility_all.to_csv(out_dir / "top_facilities.csv", index=False)
naics_pivot.reset_index().to_csv(out_dir / "naics_totals.csv", index=False)
waste_pivot.reset_index().to_csv(out_dir / "waste_totals.csv", index=False)
drivers_pivot.reset_index().to_csv(out_dir / "drivers_pivot.csv", index=False)
drivers.to_csv(out_dir / "drivers_long.csv", index=False)